# 📦 Module 6.2: Create & Run MLFlow Projects

**Goal**: Build and run an MLFlow Project for reproducible ML experiments.

---

## 1. Examine the Sample Project

Look at the `sample_project/` directory. It contains:
- `MLproject` — entry points and parameters
- `python_env.yaml` — Python environment spec
- `train.py` — the training script

In [ ]:
import os

project_dir = os.path.join(os.path.dirname(os.getcwd()), "Module_06_MLFlow_Projects", "sample_project")
if not os.path.exists(project_dir):
    project_dir = "sample_project"  # Try relative path

print("📁 Project files:")
for f in os.listdir(project_dir):
    filepath = os.path.join(project_dir, f)
    size = os.path.getsize(filepath)
    print(f"  {f} ({size} bytes)")

print("\n--- MLproject file ---")
with open(os.path.join(project_dir, "MLproject")) as f:
    print(f.read())

## 2. Run the Project Locally

Use `mlflow.run()` to execute the project. MLFlow will:
1. Read the `MLproject` file
2. Set up the environment
3. Run the entry point with the given parameters
4. Track the run in MLFlow

In [ ]:
import mlflow

# Run the project with default parameters
result = mlflow.projects.run(
    uri=project_dir,
    entry_point="train",
    env_manager="local",  # Use current environment (fastest)
    parameters={},  # Use defaults from MLproject
)

print(f"\n✅ Project run completed!")
print(f"   Run ID: {result.run_id}")

In [ ]:
# Run with custom parameters
result = mlflow.projects.run(
    uri=project_dir,
    entry_point="train",
    env_manager="local",
    parameters={
        "n_estimators": 200,
        "max_depth": 10
    },
)

print(f"\n✅ Custom parameters run completed!")
print(f"   Run ID: {result.run_id}")

## 3. Run Multiple Configurations

Projects make it easy to sweep over parameters programmatically.

In [ ]:
# Sweep over configurations
configs = [
    {"n_estimators": 50, "max_depth": 3},
    {"n_estimators": 100, "max_depth": 5},
    {"n_estimators": 200, "max_depth": 10},
]

run_ids = []
for config in configs:
    result = mlflow.projects.run(
        uri=project_dir,
        entry_point="train",
        env_manager="local",
        parameters=config,
    )
    run_ids.append(result.run_id)
    print(f"  Config {config} → Run ID: {result.run_id[:8]}...")

print(f"\n✅ Completed {len(configs)} project runs!")

In [ ]:
# Compare results
runs = mlflow.search_runs(
    filter_string=" OR ".join([f"run_id = '{rid}'" for rid in run_ids]),
    order_by=["metrics.accuracy DESC"]
)

cols = [c for c in ["tags.mlflow.runName", "params.n_estimators", "params.max_depth", 
         "metrics.accuracy", "metrics.f1_score"] if c in runs.columns]

if cols:
    print("📊 Results comparison:")
    print(runs[cols].to_string(index=False))

## 🔑 Key Takeaways

1. **MLproject file** defines entry points with typed parameters and defaults
2. **`mlflow.projects.run()`** executes a project — locally or from Git
3. **`env_manager="local"`** uses your current environment (fastest for development)
4. **`env_manager="virtualenv"`** creates an isolated environment (for reproducibility)
5. Projects are great for **parameter sweeps** — same code, different configs

---

## 🎓 Module 6 Complete!

**Next up**: Module 7 — Deployment with Docker & AWS →